# Build a Semantic Search API with Qdrant

Companion notebook for the [Build a Semantic Search API](https://qdrant.tech/documentation/tutorials-develop/neural-search/) tutorial. It searches 40,474 startup descriptions from startups-list.com by meaning: Qdrant Cloud Inference turns each description and each query into a vector with the free `sentence-transformers/all-minilm-l6-v2` model, and a FastAPI service exposes the search, with an optional city filter.

You need a free [Qdrant Cloud](https://cloud.qdrant.io) cluster: its endpoint URL and an API key.

## Setup

In [1]:
%pip install -q --disable-pip-version-check "qdrant-client>=1.19.1" fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass

for variable in ("QDRANT_URL", "QDRANT_API_KEY"):
    if not os.getenv(variable):
        os.environ[variable] = getpass(f"{variable}: ")

## Prepare sample dataset

Download the dataset and read it. Each line holds one startup with its name, description, city, and other fields.

In [3]:
!curl -sO https://storage.googleapis.com/generall-shared-data/startups_demo.json

In [4]:
import json

with open("startups_demo.json") as file:
    startups = [json.loads(line) for line in file]

print(len(startups))
startups[0]

40474


{'name': 'SaferCodes',
 'images': 'https://safer.codes/img/brand/logo-icon.png',
 'alt': 'SaferCodes Logo QR codes generator system forms for COVID-19',
 'description': 'QR codes systems for COVID-19.\nSimple tools for bars, restaurants, offices, and other small proximity businesses.',
 'link': 'https://safer.codes',
 'city': 'Chicago'}

## Upload data to Qdrant

Create a client object. With `cloud_inference=True`, the cluster generates the embeddings from the text you send.

In [5]:
import os

from qdrant_client import QdrantClient, models

client = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    cloud_inference=True,
)

Create a collection for the startup vectors. `384` is the output dimensionality of `all-MiniLM-L6-v2`.

In [6]:
if not client.collection_exists("startups"):
    client.create_collection(
        collection_name="startups",
        vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
    )

Create a payload index for the `city` field before the upload, so you can filter search results by city.

In [7]:
client.create_payload_index(
    collection_name="startups",
    field_name="city",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

Upload the data. Each `models.Document` holds the text to embed and the model name, and Qdrant Cloud turns it into a vector during the upload, which takes a few minutes.

In [8]:
EMBEDDING_MODEL = "sentence-transformers/all-minilm-l6-v2"

client.upload_collection(
    collection_name="startups",
    vectors=[
        models.Document(
            text=startup["alt"] + ". " + startup["description"],
            model=EMBEDDING_MODEL,
        )
        for startup in startups
    ],
    payload=startups,
    ids=range(len(startups)),  # Stable ids, so a rerun overwrites instead of duplicating
    batch_size=256,  # How many vectors will be uploaded in a single request?
    wait=True,  # Return only after all points are stored
)

client.count("startups").count

40474

## Build the search API

The search class from the tutorial, with the version of `search` that takes an optional city. The cell writes it to `neural_searcher.py`.

In [9]:
%%writefile neural_searcher.py
import os

from qdrant_client import QdrantClient, models

EMBEDDING_MODEL = "sentence-transformers/all-minilm-l6-v2"


class NeuralSearcher:
    def __init__(self, collection_name: str) -> None:
        self.collection_name = collection_name
        # Initialize Qdrant client, Cloud Inference embeds the query text
        self.qdrant_client = QdrantClient(
            url=os.environ["QDRANT_URL"],
            api_key=os.environ["QDRANT_API_KEY"],
            cloud_inference=True,
        )

    def search(self, text: str, city: str | None = None) -> list[dict]:
        city_filter = None
        if city:
            # Define a filter for cities
            city_filter = models.Filter(
                must=[
                    models.FieldCondition(
                        key="city",  # Store city information in a field of the same name
                        match=models.MatchValue(value=city),  # Check if the payload field has the requested value
                    )
                ]
            )

        search_result = self.qdrant_client.query_points(
            collection_name=self.collection_name,
            query=models.Document(text=text, model=EMBEDDING_MODEL),
            query_filter=city_filter,
            limit=5,
        ).points
        payloads = [hit.payload for hit in search_result]
        return payloads

Writing neural_searcher.py


Try the search directly, without and with the city filter.

In [10]:
from neural_searcher import NeuralSearcher

neural_searcher = NeuralSearcher(collection_name="startups")

[startup["name"] for startup in neural_searcher.search("cloud data platform")]

['Cloudant', 'Advanced Seismic', 'CloudStor.it', 'DataShaka', 'Crate.IO']

In [11]:
[(startup["name"], startup["city"]) for startup in neural_searcher.search("cloud data platform", city="Berlin")]

[('Crate.IO', 'Berlin'),
 ('CloudWok', 'Berlin'),
 ('CrossCloud', 'Berlin'),
 ('Productsup', 'Berlin'),
 ('infinipool', 'Berlin')]

## Deploy the search with FastAPI

The service from the tutorial. The cell writes it to `service.py`. To run the service, copy `neural_searcher.py` and `service.py` to your machine, run `python service.py` in a terminal where `QDRANT_URL` and `QDRANT_API_KEY` are set, then open http://localhost:8000/docs.

In [12]:
%%writefile service.py
from fastapi import FastAPI

# The file where NeuralSearcher is stored
from neural_searcher import NeuralSearcher

app = FastAPI()

# Create a neural searcher instance
neural_searcher = NeuralSearcher(collection_name="startups")


@app.get("/api/search")
def search_startup(q: str, city: str | None = None) -> dict:
    return {"result": neural_searcher.search(text=q, city=city)}


if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="0.0.0.0", port=8000)

Writing service.py
